# TorGen v2: Analysis Notebook

Diagnostic plots comparing generated tornado tracks against ground truth.

**Runtime:** Colab CPU (no GPU required).  
**Data:** `samples.parquet` (generated) + `.pt.gz` files (GT + weather) on Google Drive.

In [ ]:
!pip install -q git+https://github.com/jcaramichaellehigh/TorGen.git cartopy
!pip show torgen | grep -E "^(Version|Location)"

In [ ]:
import os
import gzip
import io
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch
import pyproj
import cartopy.io.shapereader as shpreader
from scipy import stats

from torgen.data.dataset import (
    _load_pt,
    coords_to_bearing_length,
    bearing_length_to_coords,
    SPLIT_YEARS,
)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# --- Paths (edit these) ---
RAW_DIR = "/content/drive/MyDrive/raw"          # .pt.gz files
SAMPLES_PATH = "/content/drive/MyDrive/checkpoints/eval/samples.parquet"
PLOTS_DIR = "/content/drive/MyDrive/checkpoints/eval/plots"

# --- NARR projection constants ---
NARR_PROJ = (
    "+proj=lcc +lon_0=-107 +lat_0=50 +x_0=5632730 +y_0=4612610 "
    "+lat_1=50 +lat_2=50 +ellps=sphere"
)
EASTING_MIN, EASTING_MAX = 5_323_932.0, 8_213_139.0
NORTHING_MIN, NORTHING_MAX = 1_817_928.0, 4_707_135.0

STP_CHANNEL = 8
EXISTS_THRESHOLD = 0.5
SPLIT = "test"

os.makedirs(PLOTS_DIR, exist_ok=True)

def _savefig(fig, name, dpi=150):
    """Save figure to Drive and display inline."""
    fig.savefig(os.path.join(PLOTS_DIR, f"{name}.png"), dpi=dpi, bbox_inches="tight")
    print(f"Saved: {name}.png")

In [ ]:
# --- State boundary helper ---
_STATE_CACHE = None

def _draw_states(ax, color="gray", linewidth=0.5, alpha=0.6):
    """Draw US state boundaries in [0,1] normalized NARR space."""
    global _STATE_CACHE
    if _STATE_CACHE is None:
        proj = pyproj.Proj(NARR_PROJ)
        shpfile = shpreader.natural_earth(
            resolution="110m", category="cultural",
            name="admin_1_states_provinces_lakes",
        )
        reader = shpreader.Reader(shpfile)
        segments = []
        e_range = EASTING_MAX - EASTING_MIN
        n_range = NORTHING_MAX - NORTHING_MIN
        for record in reader.records():
            if record.attributes.get("admin") != "United States of America":
                continue
            geom = record.geometry
            if geom.geom_type == "Polygon":
                polys = [geom]
            elif geom.geom_type == "MultiPolygon":
                polys = list(geom.geoms)
            else:
                continue
            for poly in polys:
                lons, lats = np.array(poly.exterior.coords).T
                ex, ny = proj(lons, lats)
                nx = (np.array(ex) - EASTING_MIN) / e_range
                ny_norm = (np.array(ny) - NORTHING_MIN) / n_range
                segments.append((nx, ny_norm))
        _STATE_CACHE = segments

    for nx, ny in _STATE_CACHE:
        ax.plot(nx, ny, color=color, linewidth=linewidth, alpha=alpha)

In [ ]:
# --- Load generated tracks ---
gen_df = pd.read_parquet(SAMPLES_PATH)
print(f"Generated tracks: {len(gen_df):,} rows, "
      f"{gen_df['date'].nunique()} days, "
      f"{gen_df['realization_id'].max() + 1} realizations/day")

In [ ]:
# --- Load GT tracks + weather ---
years = SPLIT_YEARS[SPLIT]
all_files = sorted(
    f for f in os.listdir(RAW_DIR)
    if (f.endswith(".pt") or f.endswith(".pt.gz")) and int(f[:4]) in years
)
print(f"Loading {len(all_files)} days from {SPLIT} split...")

gt_rows = []
day_rows = []
for fname in all_files:
    sample = _load_pt(os.path.join(RAW_DIR, fname))
    date = sample["date"]
    tracks_raw = sample["tracks"]  # (N, 6): se, sn, ee, en, width, ef
    tracks_bl = coords_to_bearing_length(tracks_raw)  # se, sn, bearing, length, width, ef
    n_tracks = tracks_bl.shape[0]
    p99_stp = float(sample["wx"][STP_CHANNEL].max())

    for i in range(n_tracks):
        t = tracks_bl[i]
        gt_rows.append({
            "date": date,
            "se": float(t[0]), "sn": float(t[1]),
            "bearing": float(t[2]), "length": float(t[3]),
            "width": float(t[4]), "ef": int(t[5]),
        })

    day_rows.append({
        "date": date,
        "gt_count": n_tracks,
        "p99_stp": p99_stp,
        "is_null": n_tracks == 0,
    })

gt_df = pd.DataFrame(gt_rows)
day_df = pd.DataFrame(day_rows)
print(f"GT tracks: {len(gt_df):,} across {len(day_df)} days "
      f"({day_df['is_null'].sum()} null days)")

# Merge generated count stats per day
gen_counts = gen_df.groupby(["date", "realization_id"]).size().reset_index(name="count")
# Include zero-count realizations for days that appear in day_df
n_realizations = gen_df["realization_id"].max() + 1
all_combos = pd.MultiIndex.from_product(
    [day_df["date"], range(n_realizations)], names=["date", "realization_id"]
).to_frame(index=False)
gen_counts = all_combos.merge(gen_counts, on=["date", "realization_id"], how="left")
gen_counts["count"] = gen_counts["count"].fillna(0).astype(int)

gen_day_stats = gen_counts.groupby("date")["count"].agg(
    mean_gen_count="mean", std_gen_count="std"
).reset_index()
day_df = day_df.merge(gen_day_stats, on="date", how="left")
day_df["std_gen_count"] = day_df["std_gen_count"].fillna(0)

In [ ]:
# --- STP deciles ---
day_df["stp_decile"] = pd.qcut(day_df["p99_stp"], 10, labels=False, duplicates="drop")

# Attach stp_decile to gen_counts for later use
gen_counts = gen_counts.merge(day_df[["date", "stp_decile", "is_null"]], on="date", how="left")

print("STP decile ranges:")
for d in sorted(day_df["stp_decile"].unique()):
    sub = day_df[day_df["stp_decile"] == d]
    print(f"  Decile {d}: STP [{sub['p99_stp'].min():.3f}, {sub['p99_stp'].max():.3f}] "
          f"({len(sub)} days)")

## Count Distribution Analysis

In [ ]:
# --- Count histograms: GT vs Generated ---
gt_day_counts = day_df["gt_count"].values
gen_real_counts = gen_counts["count"].values

max_count = max(gt_day_counts.max(), gen_real_counts.max())
bins = np.arange(-0.5, max_count + 1.5, 1)

ks_stat, ks_p = stats.ks_2samp(gt_day_counts, gen_real_counts)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(gt_day_counts, bins=bins, density=True, alpha=0.6, color="C0",
        label=f"Ground Truth (N={len(gt_day_counts)} days)")
ax.hist(gen_real_counts, bins=bins, density=True, alpha=0.6, color="C1",
        label=f"Generated (N={len(gen_real_counts):,} realizations)")
ax.set_xlabel("Tornado count per day/realization")
ax.set_ylabel("Density")
ax.set_title("Daily Tornado Count Distribution")
ax.legend()
ax.annotate(f"KS = {ks_stat:.3f}, p = {ks_p:.3g}",
            xy=(0.97, 0.95), xycoords="axes fraction",
            ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
fig.tight_layout()
_savefig(fig, "count_histogram")
plt.show()

In [ ]:
# --- Per-day count scatter with ±1σ ---
fig, ax = plt.subplots(figsize=(6, 6))
ax.errorbar(
    day_df["gt_count"], day_df["mean_gen_count"],
    yerr=day_df["std_gen_count"],
    fmt="o", markersize=3, alpha=0.5, elinewidth=0.5, capsize=0, color="C0",
)
lim = max(day_df["gt_count"].max(), day_df["mean_gen_count"].max()) + 1
ax.plot([0, lim], [0, lim], "--", color="gray", linewidth=0.8, label="1:1")
ax.set_xlim(-0.5, lim)
ax.set_ylim(-0.5, lim)
ax.set_xlabel("GT tornado count")
ax.set_ylabel("Mean generated count")
ax.set_title("Per-Day Count: GT vs Generated (±1σ)")
ax.legend()
ax.set_aspect("equal")
fig.tight_layout()
_savefig(fig, "count_scatter")
plt.show()

## Null Day Analysis by STP Decile

In [ ]:
# --- Null day fraction by STP decile ---
deciles = sorted(gen_counts["stp_decile"].dropna().unique())
null_fracs = []
nonnull_fracs = []
null_counts = []
nonnull_counts = []

for d in deciles:
    sub = gen_counts[gen_counts["stp_decile"] == d]
    # GT-null days: is_null == True
    null_sub = sub[sub["is_null"]]
    nonnull_sub = sub[~sub["is_null"]]
    # Fraction of realizations that generated zero tracks
    null_fracs.append((null_sub["count"] == 0).mean() if len(null_sub) > 0 else np.nan)
    nonnull_fracs.append((nonnull_sub["count"] == 0).mean() if len(nonnull_sub) > 0 else np.nan)
    # Sample counts (number of unique days, not realizations)
    null_counts.append(null_sub["date"].nunique())
    nonnull_counts.append(nonnull_sub["date"].nunique())

x = np.arange(len(deciles))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - w / 2, null_fracs, w, label="GT Null Days", color="C0", alpha=0.8)
bars2 = ax.bar(x + w / 2, nonnull_fracs, w, label="GT Non-Null Days", color="C1", alpha=0.8)

# Label each decile with sample counts
for i, d in enumerate(deciles):
    ax.text(i, -0.07, f"({null_counts[i]}/{nonnull_counts[i]})",
            ha="center", va="top", fontsize=7, color="gray")

ax.set_xticks(x)
ax.set_xticklabels([str(int(d)) for d in deciles])
ax.set_xlabel("STP Decile (low → high)")
ax.set_ylabel("Fraction of realizations generating zero tracks")
ax.set_title("Generated Null Rate by STP Decile")
ax.legend()
ax.set_ylim(0, 1.05)
fig.tight_layout()
_savefig(fig, "null_rate_by_stp_decile")
plt.show()

## Spatial Patterns

In [ ]:
# --- GT vs Generated genesis density ---
nbins = 20
hist_range = [[0, 1], [0, 1]]

gt_h, xedges, yedges = np.histogram2d(
    gt_df["se"].values, gt_df["sn"].values, bins=nbins, range=hist_range
)
gen_h, _, _ = np.histogram2d(
    gen_df["se"].values, gen_df["sn"].values, bins=nbins, range=hist_range
)
# Normalize generated by number of realizations so scale is comparable
gen_h = gen_h / n_realizations

vmax = max(gt_h.max(), gen_h.max())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for ax, h, title in [(ax1, gt_h, "Ground Truth"), (ax2, gen_h, "Generated (per realization)")]:
    im = ax.imshow(
        h.T, origin="lower", extent=[0, 1, 0, 1],
        vmin=0, vmax=vmax, cmap="YlOrRd", aspect="equal",
    )
    _draw_states(ax)
    ax.set_title(title)
    ax.set_xlabel("Easting (normalized)")
    ax.set_ylabel("Northing (normalized)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

fig.colorbar(im, ax=[ax1, ax2], label="Track density", shrink=0.8)
fig.suptitle("Genesis Location Density", fontsize=14, y=1.02)
fig.tight_layout()
_savefig(fig, "genesis_density")
plt.show()

In [ ]:
# --- Per-STP-decile genesis maps ---
# Merge stp_decile onto gen_df via date
gen_with_decile = gen_df.merge(day_df[["date", "stp_decile"]], on="date", how="left")

decile_vals = sorted(gen_with_decile["stp_decile"].dropna().unique())
n_deciles = len(decile_vals)
ncols, nrows = 5, 2

# Pre-compute all histograms to find shared vmax
decile_hists = {}
for d in decile_vals:
    sub = gen_with_decile[gen_with_decile["stp_decile"] == d]
    h, _, _ = np.histogram2d(sub["se"].values, sub["sn"].values, bins=20, range=[[0, 1], [0, 1]])
    n_real_d = sub.groupby("date")["realization_id"].nunique().sum()
    decile_hists[d] = h / max(n_real_d, 1)

decile_vmax = max(h.max() for h in decile_hists.values())

# STP range labels
stp_ranges = {}
for d in decile_vals:
    sub = day_df[day_df["stp_decile"] == d]
    stp_ranges[d] = (sub["p99_stp"].min(), sub["p99_stp"].max())

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
for idx, d in enumerate(decile_vals):
    r, c = divmod(idx, ncols)
    ax = axes[r, c]
    im = ax.imshow(
        decile_hists[d].T, origin="lower", extent=[0, 1, 0, 1],
        vmin=0, vmax=decile_vmax, cmap="YlOrRd", aspect="equal",
    )
    _draw_states(ax)
    lo, hi = stp_ranges[d]
    ax.set_title(f"Decile {int(d)}\nSTP [{lo:.2f}, {hi:.2f}]", fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    if c == 0:
        ax.set_ylabel("Northing")
    if r == nrows - 1:
        ax.set_xlabel("Easting")

# Hide unused axes
for idx in range(n_deciles, nrows * ncols):
    r, c = divmod(idx, ncols)
    axes[r, c].set_visible(False)

fig.colorbar(im, ax=axes, label="Track density (per realization)", shrink=0.6)
fig.suptitle("Genesis Density by STP Decile (Generated)", fontsize=14, y=1.01)
fig.tight_layout()
_savefig(fig, "genesis_by_stp_decile")
plt.show()

## Track Properties

In [ ]:
# --- Bearing distribution (rose plot) ---
# bearing_norm in [0,1] maps to [-pi, pi]: bearing = bearing_norm * 2*pi - pi
gt_bearing_rad = gt_df["bearing"].values * 2 * np.pi - np.pi
gen_bearing_rad = gen_df["bearing"].values * 2 * np.pi - np.pi

n_bins_rose = 36  # 10-degree bins
rose_bins = np.linspace(-np.pi, np.pi, n_bins_rose + 1)

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"})
# GT
gt_counts_rose, _ = np.histogram(gt_bearing_rad, bins=rose_bins)
gt_density = gt_counts_rose / gt_counts_rose.sum()
# Generated
gen_counts_rose, _ = np.histogram(gen_bearing_rad, bins=rose_bins)
gen_density = gen_counts_rose / gen_counts_rose.sum()

bin_centers = 0.5 * (rose_bins[:-1] + rose_bins[1:])
width = rose_bins[1] - rose_bins[0]

ax.bar(bin_centers, gt_density, width=width, alpha=0.5, color="C0", label="GT")
ax.bar(bin_centers, gen_density, width=width, alpha=0.5, color="C1", label="Generated")
# 0 = North (up), bearing is compass-style: atan2(dx, dy)
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)  # clockwise
ax.set_title("Track Bearing Distribution", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
fig.tight_layout()
_savefig(fig, "bearing_rose")
plt.show()

In [ ]:
# --- Length distribution ---
ks_len, ks_len_p = stats.ks_2samp(gt_df["length"].values, gen_df["length"].values)

fig, ax = plt.subplots(figsize=(8, 4))
bins_len = np.linspace(0, max(gt_df["length"].max(), gen_df["length"].max()), 50)
ax.hist(gt_df["length"].values, bins=bins_len, density=True, alpha=0.6,
        color="C0", label=f"GT (N={len(gt_df):,})")
ax.hist(gen_df["length"].values, bins=bins_len, density=True, alpha=0.6,
        color="C1", label=f"Generated (N={len(gen_df):,})")
ax.set_xlabel("Track length (normalized)")
ax.set_ylabel("Density")
ax.set_title("Track Length Distribution")
ax.legend()
ax.annotate(f"KS = {ks_len:.3f}, p = {ks_len_p:.3g}",
            xy=(0.97, 0.95), xycoords="axes fraction",
            ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
fig.tight_layout()
_savefig(fig, "length_distribution")
plt.show()

In [ ]:
# --- Width distribution ---
ks_wid, ks_wid_p = stats.ks_2samp(gt_df["width"].values, gen_df["width"].values)

fig, ax = plt.subplots(figsize=(8, 4))
bins_wid = np.linspace(0, max(gt_df["width"].max(), gen_df["width"].max()), 50)
ax.hist(gt_df["width"].values, bins=bins_wid, density=True, alpha=0.6,
        color="C0", label=f"GT (N={len(gt_df):,})")
ax.hist(gen_df["width"].values, bins=bins_wid, density=True, alpha=0.6,
        color="C1", label=f"Generated (N={len(gen_df):,})")
ax.set_xlabel("Track width (normalized)")
ax.set_ylabel("Density")
ax.set_title("Track Width Distribution")
ax.legend()
ax.annotate(f"KS = {ks_wid:.3f}, p = {ks_wid_p:.3g}",
            xy=(0.97, 0.95), xycoords="axes fraction",
            ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
fig.tight_layout()
_savefig(fig, "width_distribution")
plt.show()

In [ ]:
# --- EF rating distribution ---
ef_classes = list(range(6))
gt_ef_counts = gt_df["ef"].value_counts().reindex(ef_classes, fill_value=0)
gen_ef_counts = gen_df["ef"].value_counts().reindex(ef_classes, fill_value=0)

gt_ef_frac = gt_ef_counts / gt_ef_counts.sum()
gen_ef_frac = gen_ef_counts / gen_ef_counts.sum()

# Chi-squared test
chi2, chi2_p = stats.chisquare(gen_ef_counts.values, f_exp=gt_ef_frac.values * gen_ef_counts.sum())

x = np.arange(len(ef_classes))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars_gt = ax.bar(x - w / 2, gt_ef_frac.values, w, label="GT", color="C0", alpha=0.8)
bars_gen = ax.bar(x + w / 2, gen_ef_frac.values, w, label="Generated", color="C1", alpha=0.8)

# Label bars with percentages
for bars in [bars_gt, bars_gen]:
    for bar in bars:
        h = bar.get_height()
        if h > 0.005:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                    f"{h * 100:.1f}%", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([f"EF{c}" for c in ef_classes])
ax.set_xlabel("EF Rating")
ax.set_ylabel("Fraction")
ax.set_title("EF Rating Distribution")
ax.legend()
ax.annotate(f"χ² = {chi2:.1f}, p = {chi2_p:.3g}",
            xy=(0.97, 0.95), xycoords="axes fraction",
            ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
fig.tight_layout()
_savefig(fig, "ef_distribution")
plt.show()